# PyTorch Tensors - the bricks under every model

*ML & NLP course - Data Trainers LLC - Axel Sirota*

## Where we are

In Part A you shipped real NLP with zero training: a `pipeline()` for every task and a zero-shot router that hit roughly 70-85 percent accuracy. We kept promising one thing: to push past that ceiling you have to train a model, and to train a model you have to speak PyTorch. This is where we open the black box.

We are not doing a deep PyTorch course. We are picking up exactly the bricks the next three notebooks snap together - tensors, shapes, broadcasting, and autograd - and nothing more.

## Learning objectives

By the end of this notebook you will be able to:

1. Create tensors of any shape and dtype with `torch.tensor`, `torch.zeros/ones/rand/randn`, and convert to and from NumPy.
2. Run arithmetic, reductions, ReLU, and matrix multiply on tensors.
3. Reshape tensors with `reshape/view/transpose/permute/squeeze/unsqueeze`, and know when `view` fails.
4. Index, slice, boolean-mask, and apply broadcasting confidently.
5. Compute gradients with `requires_grad=True` and `.backward()` - the engine that trains every model in Part B and Part C, and the PyTorch replacement for `tf.GradientTape`.

## Prerequisites

- Comfortable with NumPy arrays and basic ML.
- No prior PyTorch needed - that is the point.

## How to run

- Colab (recommended): Runtime -> Change runtime type -> GPU is nice but not required.
- Everything here is tiny and runs fine on CPU.

Let's pick up the bricks.

## Section 0 - Environment Setup

Colab already ships PyTorch and NumPy, so the install cell is mostly a safety net. We pin `numpy<2` to stay consistent with the rest of the course (gensim and friends in B5 need it). Then we import, set seeds, and detect the device.

In [ ]:
# Install required packages (run this first on Google Colab).
# Colab already ships a recent PyTorch, so we only pin NumPy to <2 for course-wide consistency.
# NOTE: Colab preinstalls numpy 2.x. After this cell runs, if Colab shows a
# "RESTART RUNTIME" button, click it, then run the notebook again from the top.
!pip install -q "numpy<2"

In [ ]:
import numpy as np
import torch
import torch.nn as nn  # we only use nn for relu/Linear demos later; full nn comes in B6

# ---- Reproducibility (same SEED=42 we use course-wide) ----
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ---- Device detection: GPU if available, else CPU ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version : {torch.__version__}")
print(f"NumPy version   : {np.__version__}  (expected 1.x because we pinned numpy<2)")
print(f"Using device    : {device}")
if device.type == 'cuda':
    print(f"GPU name        : {torch.cuda.get_device_name(0)}")

print("\nEnvironment setup complete.")

## What are we building toward?

Nothing fancy in this notebook - on purpose. The payoff is later, and it helps to see the chain now:

- Today (B4): a tensor is a NumPy array plus GPU and autograd. You will differentiate a small expression by hand.
- B5: every sentence becomes a fixed 384-dimensional float vector (from the `all-MiniLM-L6-v2` embedder you met in A1). A vector is just a 1-D tensor.
- B6: those vectors flow through `nn.Linear` layers - which are matrix multiply plus a broadcasted bias add, both of which you will write today.
- B7 (the stopper): you train a small MLP on those 384-dim tensors. Training is just calling `.backward()` in a loop - the exact mechanic from Section 5 below.
- C9: the same `.backward()` fine-tunes a whole transformer for the chatbot.

So when you call `loss.backward()` near the end of this notebook, you are running the literal engine that trains everything that follows. Let's earn it.

**The B4 to chatbot chain: every brick snaps into the next.**

```mermaid
graph TD
    A[B4 today: tensors plus autograd] --> B[B5: sentence to 384-dim vector]
    B --> C[B6: vectors through nn.Linear layers]
    C --> D[B7 stopper: train MLP head]
    D --> E[C9: backward fine-tunes DistilBERT]
    E --> F[Gradio chatbot]
    A -.-> G[loss.backward is the engine]
    G -.-> D
```


## Section 1 - Tensor Basics

Mental model: a PyTorch tensor is a NumPy array, plus optional GPU placement, plus optional autograd tracking. The factory functions are almost a 1-to-1 map from NumPy:

| NumPy | PyTorch |
|-------|---------|
| `np.array([1, 2, 3])` | `torch.tensor([1, 2, 3])` |
| `np.zeros((3, 3))` | `torch.zeros(3, 3)` |
| `np.ones((2, 4))` | `torch.ones(2, 4)` |
| `np.random.rand(2, 3)` | `torch.rand(2, 3)` |
| `np.random.randn(2, 3)` | `torch.randn(2, 3)` |

Two attributes you will check constantly:

```python
t = torch.tensor([1.0, 2.0, 3.0])
t.shape   # torch.Size([3])
t.dtype   # torch.float32
```

NumPy and PyTorch convert both ways:

```python
arr  = np.array([1.0, 2.0])
t    = torch.from_numpy(arr)   # SHARES memory: mutating arr also changes t
t2   = torch.tensor(arr)       # COPIES: independent of arr
back = t.numpy()               # back to NumPy (CPU tensors only)
```

The shared-memory detail of `from_numpy` is a classic source of surprise bugs - know it exists.

### Demo - factory functions in action

**Mental model: a tensor is a NumPy array with two superpowers.**

```mermaid
graph TD
    A[NumPy ndarray] --> B[PyTorch tensor]
    B --> C[Same shape and dtype API]
    B --> D[Optional GPU placement]
    B --> E[Optional autograd tracking]
    D --> F[Runs on cuda device]
    E --> G[backward computes grad]
    C --> H[Drop-in for NumPy math]
```


In [ ]:
# Demo: tensor creation and inspection
scalar  = torch.tensor(3.14)                       # 0-D tensor
vector  = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])  # 1-D
matrix  = torch.zeros(3, 3)                         # 2-D all zeros
rand3d  = torch.randn(2, 3, 4)                      # 3-D standard normal

for name, t in [('scalar', scalar), ('vector', vector),
                ('matrix', matrix), ('rand3d', rand3d)]:
    print(f"{name:8s}  shape={str(t.shape):20s}  dtype={t.dtype}")

# NumPy round-trip
arr  = np.array([[1.0, 2.0], [3.0, 4.0]])
t_np = torch.from_numpy(arr)     # zero-copy view: shares memory with arr
back = t_np.numpy()              # back to numpy
print(f"\nNumPy -> torch -> NumPy: arr type={type(arr).__name__}, "
      f"t_np type={type(t_np).__name__}, back type={type(back).__name__}")
print(f"All values identical: {np.allclose(arr, back)}")

### Lab 1 - Tensor Basics

Complete the tasks below. Verification code is provided so you can check yourself immediately.

Tasks:
1. Create a scalar tensor with the value `7.0` and name it `s`.
2. Create a 1-D tensor with the five elements `[10, 20, 30, 40, 50]` as `torch.float32`, named `v`.
3. Create a 2-D tensor of shape `(3, 3)` filled with ones, named `m`.
4. Create a 3-D tensor of shape `(2, 4, 4)` filled with random normal values, named `t3d`.
5. Convert the NumPy array `np_arr` (provided in the starter) to a PyTorch tensor named `t_from_np`.
6. Convert `t_from_np` back to a NumPy array named `arr_back`.

In [ ]:
np_arr = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])

# 1. Scalar tensor holding the value 7.0
s = None  # YOUR CODE

# 2. 1-D tensor [10, 20, 30, 40, 50] as float32
v = None  # YOUR CODE

# 3. 2-D tensor of ones, shape (3, 3)
m = None  # YOUR CODE

# 4. 3-D tensor of random normal values, shape (2, 4, 4)
t3d = None  # YOUR CODE

# 5. Convert np_arr to a PyTorch tensor
t_from_np = None  # YOUR CODE

# 6. Convert t_from_np back to a NumPy array
arr_back = None  # YOUR CODE

# ---- Verification ----
if s is not None:
    print(f"s          : value={s.item():.1f}  shape={s.shape}  dtype={s.dtype}")
if v is not None:
    print(f"v          : shape={v.shape}  dtype={v.dtype}  values={v.tolist()}")
if m is not None:
    print(f"m          : shape={m.shape}  all_ones={m.all().item()}")
if t3d is not None:
    print(f"t3d        : shape={t3d.shape}  dtype={t3d.dtype}")
if t_from_np is not None:
    print(f"t_from_np  : shape={t_from_np.shape}  dtype={t_from_np.dtype}")
if arr_back is not None:
    print(f"arr_back   : type={type(arr_back).__name__}  shape={arr_back.shape}")

## Section 2 - Tensor Operations

PyTorch operators map almost identically to NumPy - the difference is they run on the GPU and can track gradients.

Arithmetic (`+ - * /`) is element-wise. There are named equivalents `torch.add/sub/mul/div`.

Statistics:
```python
t.mean()        # scalar mean over all elements
t.mean(dim=0)   # mean along dim 0 (collapse rows -> one value per column)
t.std()         # standard deviation
```

Activation:
```python
torch.relu(t)   # max(0, x) element-wise, no import needed
```

Matrix multiply:
```python
A @ B           # same as torch.matmul(A, B)
torch.mm(A, B)  # strict 2-D only
```

Matrix multiply (`@`) is the heartbeat of a neural network layer: an `nn.Linear` is literally `x @ W.T + b`. You will recognize it again in B6.

### Demo - operations

In [ ]:
torch.manual_seed(SEED)

a = torch.tensor([1.0, 2.0, 3.0, 4.0])
b = torch.tensor([10.0, 20.0, 30.0, 40.0])

print("a + b  =", (a + b).tolist())
print("a - b  =", (a - b).tolist())
print("a * b  =", (a * b).tolist())
print("a / b  =", (a / b).tolist())

print(f"\na.mean() = {a.mean().item():.4f}")
print(f"a.std()  = {a.std().item():.4f}")

# ReLU on a tensor with negatives: clamps everything below 0 to 0
c = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])
print(f"\ntorch.relu({c.tolist()}) = {torch.relu(c).tolist()}")

# Matrix multiply: (2,3) @ (3,2) -> (2,2)
A = torch.ones(2, 3)
B = torch.ones(3, 2) * 2.0
C = A @ B
print(f"\n(2x3) @ (3x2) = {C.shape}, values:\n{C}")

### Lab 2 - Tensor Operations

Tasks:
1. With `x = [1.0, 4.0, 9.0, 16.0]` and `y = [2.0, 2.0, 3.0, 4.0]`, compute element-wise add, subtract, multiply, and divide into `add_xy`, `sub_xy`, `mul_xy`, `div_xy`.
2. Build a `(4, 4)` random normal tensor `mat`; compute its global mean (`mat_mean`) and std (`mat_std`).
3. Apply ReLU to `torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])` into `relu_out`.
4. With `P = torch.ones(3, 4)` and `Q = torch.ones(4, 5) * 3.0`, compute the matrix product `PQ`.

In [ ]:
torch.manual_seed(SEED)

# 1. Element-wise arithmetic
x = torch.tensor([1.0, 4.0, 9.0, 16.0])
y = torch.tensor([2.0, 2.0, 3.0,  4.0])

add_xy = None  # YOUR CODE
sub_xy = None  # YOUR CODE
mul_xy = None  # YOUR CODE
div_xy = None  # YOUR CODE

# 2. Mean and std of a (4,4) random matrix
mat = torch.randn(4, 4)
mat_mean = None  # YOUR CODE
mat_std  = None  # YOUR CODE

# 3. ReLU applied to torch.tensor([-2., -1., 0., 1., 2.])
relu_out = None  # YOUR CODE

# 4. Matrix product of P and Q
P  = torch.ones(3, 4)
Q  = torch.ones(4, 5) * 3.0
PQ = None  # YOUR CODE

# ---- Verification ----
if add_xy is not None:
    print(f"x + y  = {add_xy.tolist()}")
    print(f"x - y  = {sub_xy.tolist()}")
    print(f"x * y  = {mul_xy.tolist()}")
    print(f"x / y  = {div_xy.tolist()}")
if mat_mean is not None:
    print(f"\nmat mean = {mat_mean.item():.4f}  std = {mat_std.item():.4f}")
if relu_out is not None:
    print(f"relu_out = {relu_out.tolist()}")
if PQ is not None:
    print(f"PQ shape = {PQ.shape}  (expected (3, 5))  all entries 12.0: {(PQ == 12.0).all().item()}")

## Section 3 - Manipulating Shapes

Reshaping is the plumbing of model wiring: one layer outputs `(B, T, D)` and the next wants `(B, T*D)`. You will do this constantly.

| Method | Notes |
|--------|-------|
| `.reshape(new_shape)` | Returns a view if possible, else a copy. Safe general purpose. |
| `.view(new_shape)` | Always a view, but the tensor MUST be contiguous in memory. Faster, stricter. |
| `.transpose(d0, d1)` | Swaps two dims. Returns a view (and makes the tensor non-contiguous). |
| `.permute(*dims)` | Reorders all dims at once. Returns a view. |
| `.flatten(start, end)` | Collapses a range of dims into one. |
| `.squeeze(dim)` | Removes size-1 dims. No `dim` removes all of them. |
| `.unsqueeze(dim)` | Inserts a new size-1 dim at `dim`. |

The number-one reshape gotcha: `.view()` fails after `.transpose()` or `.permute()`, because those return a non-contiguous view and `.view()` needs contiguous memory. PyTorch raises a RuntimeError that literally tells you to use `.reshape()`. Two fixes:

```python
t2 = t.transpose(0, 1)
t2.view(-1)                 # RuntimeError: not compatible with size/stride
t2.reshape(-1)              # works: reshape copies if it has to
t2.contiguous().view(-1)    # also works: force a contiguous copy first
```

Rule of thumb: reach for `.reshape()` unless you have a reason to want `.view()`.

Common pattern - adding a batch dimension:
```python
x = torch.randn(28, 28)   # one image
x = x.unsqueeze(0)        # (1, 28, 28) - batch of 1
```

### Demo - shape manipulation

**Choosing view vs reshape: contiguity is the deciding factor.**

```mermaid
flowchart LR
    A[Need a new shape] --> B{Is tensor contiguous?}
    B -->|Yes| C[view works, fast no copy]
    B -->|No, e.g. after transpose| D[view raises RuntimeError]
    D --> E[reshape: copies if needed]
    D --> F[contiguous then view]
    A --> G[Rule of thumb: prefer reshape]
```


In [ ]:
t = torch.arange(16, dtype=torch.float32)  # [0, 1, ..., 15]
print("Original shape:", t.shape)

r1 = t.reshape(4, 4)
print("reshape(4,4)   :", r1.shape)

r2 = t.reshape(2, 8)
print("reshape(2,8)   :", r2.shape)

# transpose swaps two dims (and yields a non-contiguous view)
r3 = r1.transpose(0, 1)
print("transpose(0,1) :", r3.shape, "  (rows become columns)")

# permute on a 3-D tensor
t3 = torch.randn(2, 3, 5)
r4 = t3.permute(2, 0, 1)   # (2,3,5) -> (5,2,3)
print(f"permute(2,0,1) on {t3.shape}: {r4.shape}")

# flatten and squeeze/unsqueeze
t4 = torch.zeros(2, 1, 3, 1)
print(f"\nsqueeze()      : {t4.shape} -> {t4.squeeze().shape}")
print(f"unsqueeze(0)   : {t.shape} -> {t.unsqueeze(0).shape}")
print(f"flatten()      : {r1.shape} -> {r1.flatten().shape}")

# The classic gotcha: view() on a transposed (non-contiguous) tensor fails.
print("\n--- view vs reshape after transpose ---")
try:
    bad = r3.view(-1)
except RuntimeError as e:
    print(f"r3.view(-1) raised: {type(e).__name__} (non-contiguous)")
print("r3.reshape(-1) works:", r3.reshape(-1).shape)

### Lab 3 - Manipulating Shapes

Tasks:
1. Create `base = torch.arange(24, dtype=torch.float32)`. Reshape it to `(3, 8)` -> `reshaped_3x8`.
2. Reshape `base` to `(2, 3, 4)` -> `reshaped_3d`.
3. Transpose `reshaped_3x8` (swap dim 0 and dim 1) -> `transposed`.
4. Permute `reshaped_3d` from `(2, 3, 4)` to `(4, 2, 3)` -> `permuted`.
5. Flatten `reshaped_3d` entirely -> `flat`.
6. Start with `img = torch.randn(28, 28)`. Add a batch dimension at position 0 -> `img_batched` (shape should be `(1, 28, 28)`).
7. Remove that batch dimension -> `img_back` (shape should be `(28, 28)`).

In [ ]:
base = torch.arange(24, dtype=torch.float32)
img  = torch.randn(28, 28)

# 1. Reshape to (3, 8)
reshaped_3x8 = None  # YOUR CODE

# 2. Reshape to (2, 3, 4)
reshaped_3d  = None  # YOUR CODE

# 3. Transpose reshaped_3x8 (swap dim 0 and dim 1)
transposed   = None  # YOUR CODE

# 4. Permute reshaped_3d to (4, 2, 3)
permuted     = None  # YOUR CODE

# 5. Flatten reshaped_3d entirely
flat         = None  # YOUR CODE

# 6. Add a batch dim to img -> (1, 28, 28)
img_batched  = None  # YOUR CODE

# 7. Remove that batch dim -> (28, 28)
img_back     = None  # YOUR CODE

# ---- Verification ----
checks = [
    ('reshaped_3x8', reshaped_3x8, (3, 8)),
    ('reshaped_3d',  reshaped_3d,  (2, 3, 4)),
    ('transposed',   transposed,   (8, 3)),
    ('permuted',     permuted,     (4, 2, 3)),
    ('flat',         flat,         (24,)),
    ('img_batched',  img_batched,  (1, 28, 28)),
    ('img_back',     img_back,     (28, 28)),
]
for name, t, expected in checks:
    if t is not None:
        status = "OK" if tuple(t.shape) == expected else f"WRONG (got {tuple(t.shape)})"
        print(f"{name:15s} shape={tuple(t.shape)}  expected={expected}  [{status}]")

## Section 4 - Indexing and Broadcasting

These two go together: you slice a tensor to pull out the rows you want, and you broadcast to combine tensors of different shapes without copying.

### Indexing - identical to NumPy

```python
t = torch.tensor([[1, 2, 3],
                  [4, 5, 6],
                  [7, 8, 9]])

t[0]        # first row -> tensor([1, 2, 3])
t[0, 2]     # row 0, col 2 -> tensor(3)
t[1:3, :]   # rows 1 and 2, all columns
t[:, 1]     # all rows, column 1
t[t > 4]    # boolean mask: every element greater than 4
```

Boolean masking is everywhere in NLP: dropping padding tokens, keeping high-confidence predictions, filtering embeddings.

### Broadcasting - the rules

Broadcasting lets PyTorch combine different shapes without copying data. Same rules as NumPy:

1. If the tensors have different numbers of dims, prepend 1s to the smaller shape.
2. Dimensions of size 1 are stretched to match the other tensor.
3. If sizes still disagree after rules 1 and 2, you get an error.

```
(3, 4) + (4,)   -> prepend 1 -> (1, 4) -> stretch -> (3, 4)   OK
(3, 1) + (1, 4) -> stretch both -> (3, 4)                     OK
(3, 4) + (3, 3) -> 4 vs 3 cannot align                        ERROR
```

This is exactly how a bias add works inside `nn.Linear`: the bias is `(out_features,)` and gets broadcast across every row of the `(batch, out_features)` output. You are previewing B6 plumbing.

### Demo - indexing and broadcasting

**How broadcasting aligns two shapes step by step.**

```mermaid
flowchart LR
    A[Two tensors, different shapes] --> B[Rule 1: prepend 1s to smaller shape]
    B --> C[Rule 2: stretch size-1 dims to match]
    C --> D{All dims now agree?}
    D -->|Yes| E[Broadcast OK, no data copied]
    D -->|No| F[RuntimeError: cannot align]
    E --> G[Example: 3x4 plus 4 is bias add]
```


In [ ]:
# ---- Indexing ----
grid = torch.arange(1, 10, dtype=torch.float32).reshape(3, 3)
print("grid:\n", grid)
print("\ngrid[0]        =", grid[0].tolist(),   "  (first row)")
print("grid[0, 2]     =", grid[0, 2].item(),    "  (row 0, col 2)")
print("grid[1:3, :]   =\n", grid[1:3, :])
print("grid[:, 1]     =", grid[:, 1].tolist(),  "  (middle column)")

mask = grid > 4
print("\ngrid > 4 mask:\n", mask)
print("grid[grid > 4] =", grid[grid > 4].tolist())  # boolean masking

# ---- Broadcasting ----
print("\n--- broadcasting ---")
# Case 1: (3, 4) + (4,) -> (3, 4). This is the bias-add pattern.
M = torch.ones(3, 4)
row = torch.tensor([0.0, 1.0, 2.0, 3.0])    # shape (4,)
print("(3,4) + (4,) ->", (M + row).shape)
print(M + row)

# Case 2: (3, 1) + (1, 4) -> (3, 4). Both dims stretch (outer-product shape).
col  = torch.tensor([[10.0], [20.0], [30.0]])   # (3, 1)
row2 = torch.tensor([[1.0, 2.0, 3.0, 4.0]])     # (1, 4)
print("\n(3,1) + (1,4) ->", (col + row2).shape)
print(col + row2)

# Case 3: intentional failure to read the error message
try:
    bad = torch.ones(3, 4) + torch.ones(3, 3)
except RuntimeError as e:
    print(f"\nExpected error (4 vs 3 cannot broadcast): {type(e).__name__}")

### Lab 4 - Indexing and Broadcasting

Use `data` and the broadcasting tensors created in the starter.

Core tasks:
1. Extract the element at row 1, column 2 -> `elem`.
2. Extract rows 1 and 2 (all columns) -> `rows_1_2`.
3. Extract the last column (all rows) -> `last_col`.
4. Boolean-mask all elements less than 0 -> `negatives`.
5. Add the bias vector `bias` (shape `(5,)`) to every row of `features` (shape `(6, 5)`) -> `biased`. The result should be `(6, 5)`.
6. Multiply the column vector `col_scale` (shape `(3, 1)`) by the row vector `row_weights` (shape `(1, 3)`) -> `scaled`. The result should be `(3, 3)`.

Stretch (fast finishers): build a pairwise distance matrix without any loop. Given `pts` of shape `(N, D)`, compute `dists` of shape `(N, N)` where `dists[i, j]` is the Euclidean distance between row i and row j. Hint about the SHAPES only (not the call): you want to subtract a `(N, 1, D)`-shaped view from a `(1, N, D)`-shaped view so broadcasting produces an `(N, N, D)` difference, then reduce the last dimension. This is the exact trick behind the sentence-similarity work in B5.

In [ ]:
torch.manual_seed(SEED)

# ---- Indexing data ----
data = torch.randn(4, 5) * 5   # (4, 5) values roughly in [-15, 15]
print("data:\n", data.round(decimals=2))

# 1. Element at row 1, column 2
elem      = None  # YOUR CODE

# 2. Rows 1 and 2, all columns
rows_1_2  = None  # YOUR CODE

# 3. Last column (all rows)
last_col  = None  # YOUR CODE

# 4. All elements less than 0 (boolean mask)
negatives = None  # YOUR CODE

# ---- Broadcasting data ----
features    = torch.randn(6, 5)
bias        = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])     # (5,)
col_scale   = torch.tensor([[2.0], [3.0], [4.0]])         # (3, 1)
row_weights = torch.tensor([[1.0, 0.5, 0.25]])            # (1, 3)

# 5. Add bias to every row of features -> (6, 5)
biased = None  # YOUR CODE

# 6. Combine col_scale and row_weights so the result is (3, 3)
scaled = None  # YOUR CODE

# ---- Stretch: pairwise distance matrix (no loops) ----
# Goal: dists[i, j] = Euclidean distance between pts[i] and pts[j]. Final shape (N, N).
pts   = torch.randn(5, 3)   # N=5 points in D=3 dimensions
dists = None  # YOUR CODE  (shape (5, 5); the diagonal should be ~0)

# ---- Verification ----
if elem is not None:
    print(f"\nelem      = {elem.item():.4f}")
if rows_1_2 is not None:
    print(f"rows_1_2  shape={rows_1_2.shape}  (expected (2, 5))")
if last_col is not None:
    print(f"last_col  shape={last_col.shape}  (expected (4,))")
if negatives is not None:
    print(f"negatives = {negatives.round(decimals=2).tolist()}")
if biased is not None:
    print(f"biased    shape={biased.shape}  (expected (6, 5))")
if scaled is not None:
    print(f"scaled    shape={scaled.shape}  (expected (3, 3))\n{scaled}")
if dists is not None:
    print(f"dists     shape={dists.shape}  (expected (5, 5))")
    print(f"diagonal ~0: {torch.allclose(dists.diagonal(), torch.zeros(5), atol=1e-4)}")

## Section 5 - Autograd and Differentiation

This is where PyTorch leaves NumPy behind. Autograd is the engine that makes backpropagation automatic - and it is the entire reason the rest of this course works.

The idea: mark a tensor with `requires_grad=True` and PyTorch records every operation on it in a computation graph. Call `.backward()` on a scalar loss and gradients flow backward through that graph by the chain rule, landing in each tensor's `.grad`.

```python
w = torch.tensor(3.0, requires_grad=True)
loss = w ** 2          # builds the graph loss = w^2
loss.backward()        # d(loss)/d(w) = 2w = 6
print(w.grad)          # tensor(6.)
```

Three rules worth burning in now, because every one of them bites beginners:

1. `.backward()` needs a SCALAR. If your output is a vector or matrix, reduce it first with `.mean()` or `.sum()`, otherwise PyTorch raises "grad can be implicitly created only for scalar outputs."
2. Gradients ACCUMULATE. Each `.backward()` ADDS into `.grad`. In a training loop you must zero them between steps (an optimizer does this for you via `zero_grad()`; doing it by hand you call `.grad.zero_()`).
3. Only floating-point tensors can require gradients. `torch.tensor([1, 2, 3], requires_grad=True)` errors because it is integer - use `dtype=torch.float32`.

Coming from TensorFlow, this replaces `tf.GradientTape`:

```python
# TF2
with tf.GradientTape() as tape:
    loss = w ** 2
grad = tape.gradient(loss, w)

# PyTorch - no context manager needed
loss = w ** 2
loss.backward()
grad = w.grad
```

And `torch.no_grad()` switches tracking off for inference, to save memory and time:

```python
with torch.no_grad():
    predictions = model(x)   # no graph built
```

### Demo - autograd

**The autograd loop: forward records, backward fills grad.**

```mermaid
graph TD
    A[w with requires_grad True] --> B[Forward op: loss equals w squared]
    B --> C[Autograd records graph]
    C --> D[Call loss.backward on scalar]
    D --> E[Chain rule flows backward]
    E --> F[w.grad filled with 2w]
    F --> G[Update w under no_grad]
    G --> H[Zero grad, repeat next step]
```


In [ ]:
# Demo 1: simple scalar gradient
w = torch.tensor(3.0, requires_grad=True)
loss = w ** 2          # loss = w^2, so d/dw = 2w
loss.backward()
print(f"w = {w.item()},  loss = {loss.item()},  grad = {w.grad.item()}")
print(f"Expected grad: 2 * w = {2 * w.item()}")

# Demo 2: gradient of a two-variable function
# f(x, y) = x^2 + 3*x*y   at x=2, y=5
# df/dx = 2x + 3y = 19,  df/dy = 3x = 6
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(5.0, requires_grad=True)
f = x**2 + 3 * x * y
f.backward()
print(f"\nf(2,5) = {f.item()},  df/dx = {x.grad.item()},  df/dy = {y.grad.item()}")
print(f"Expected: df/dx = {2*x.item() + 3*y.item()},  df/dy = {3*x.item()}")

# Demo 3: torch.no_grad() disables tracking (inference mode)
with torch.no_grad():
    z = w ** 3
    print(f"\nz.requires_grad = {z.requires_grad}  (should be False)")

### Lab 5 - Autograd (core + stretch + homework)

Core tasks:
1. Create a scalar tensor `a` holding `4.0` that tracks gradients. Compute `loss_a = a ** 3`, call backward, and store the gradient in `grad_a`. It should equal `3 * a^2 = 48`.
2. Create scalar tensors `p = 2.0` and `q = -3.0` that both track gradients. Compute `loss_pq = p * q + p ** 2`, call backward, and store the gradients in `grad_p` and `grad_q`. Check them against the calculus: d/dp = q + 2p, d/dq = p.
3. Recompute `p * q + p ** 2` inside a no-grad context into `loss_no_grad`, and confirm that `loss_no_grad.requires_grad` is `False`.

Stretch (in-notebook, fast finishers) - gradient descent by hand:
Fit `y = 2x + 1` from noisy data WITHOUT an optimizer, so you see what `optimizer.step()` will automate in B6. Initialize `w` and `b` to track gradients, then loop: compute the prediction, compute MSE loss, call backward, and update the parameters inside a no-grad block. Remember rule 2 from the theory - you must zero the gradients each step or they pile up. After ~200 steps `w` should be near 2.0 and `b` near 1.0.

Homework extension (async, production-grade) - gradient checking:
Real teams verify a hand-written backward pass against a numerical gradient before trusting it. Take `f(x) = x**3 + 2*x` at `x = 1.5`. Compute the autograd gradient with `.backward()`. Then compute the numerical gradient with the central finite-difference formula `(f(x+eps) - f(x-eps)) / (2*eps)` for a small `eps` (use `dtype=torch.float64` for precision). Confirm the two agree to within `1e-4`. This is exactly the idea behind `torch.autograd.gradcheck`, the tool you would use when writing a custom autograd Function. Write it up in your own notebook cell.

In [ ]:
# ---- Core ----

# 1. Gradient of a^3 at a = 4.0
a      = None  # YOUR CODE: a scalar tensor 4.0 that tracks gradients
loss_a = None  # YOUR CODE: a cubed
# YOUR CODE: trigger backpropagation on loss_a
grad_a = None  # YOUR CODE: read the gradient of a

# 2. Two-variable gradient at p = 2.0, q = -3.0
p       = None  # YOUR CODE: scalar tensor 2.0 that tracks gradients
q       = None  # YOUR CODE: scalar tensor -3.0 that tracks gradients
loss_pq = None  # YOUR CODE: p*q + p squared
# YOUR CODE: trigger backpropagation on loss_pq
grad_p  = None  # YOUR CODE: gradient of p
grad_q  = None  # YOUR CODE: gradient of q

# 3. Same expression as task 2, but with gradient tracking switched off
loss_no_grad = None  # YOUR CODE: recompute the task-2 expression inside a no-grad context

# ---- Verification (core) ----
if grad_a is not None:
    expected_a = 3 * (4.0 ** 2)
    print(f"grad_a = {grad_a.item():.1f}  expected = {expected_a:.1f}  "
          f"OK={abs(grad_a.item()-expected_a)<1e-4}")
if grad_p is not None and grad_q is not None:
    exp_p = -3.0 + 2 * 2.0   # q + 2p = 1
    exp_q = 2.0              # p = 2
    print(f"grad_p = {grad_p.item():.1f}  expected = {exp_p:.1f}  OK={abs(grad_p.item()-exp_p)<1e-4}")
    print(f"grad_q = {grad_q.item():.1f}  expected = {exp_q:.1f}  OK={abs(grad_q.item()-exp_q)<1e-4}")
if loss_no_grad is not None:
    print(f"loss_no_grad.requires_grad = {loss_no_grad.requires_grad}  (should be False)")

# ---- Stretch: gradient descent by hand ----
torch.manual_seed(SEED)
lr, STEPS = 0.1, 200
x_gd = torch.linspace(-1, 1, 50)
y_gd = 2 * x_gd + 1 + 0.1 * torch.randn(50)   # ground truth y = 2x + 1 + noise

w_gd = None  # YOUR CODE: scalar 0.0 that tracks gradients
b_gd = None  # YOUR CODE: scalar 0.0 that tracks gradients

for step in range(STEPS):
    if w_gd is None or b_gd is None:
        break
    y_pred_gd = None  # YOUR CODE: predict w_gd * x_gd + b_gd
    loss_gd   = None  # YOUR CODE: mean squared error between y_pred_gd and y_gd
    # YOUR CODE: backpropagate loss_gd
    with torch.no_grad():
        pass  # YOUR CODE: step w_gd and b_gd down their gradients by lr
    # YOUR CODE: zero the gradients of w_gd and b_gd for the next step

if w_gd is not None and w_gd.grad is not None:
    print(f"\nLearned w = {w_gd.item():.4f}  (expected ~2.0)")
    print(f"Learned b = {b_gd.item():.4f}  (expected ~1.0)")

## Wrap-up: you built the bricks

Here is what you now have in your hands:

| Section | Core skill |
|---------|-----------|
| 1 - Basics | `torch.tensor`, factory functions, `.shape`, `.dtype`, numpy round-trip |
| 2 - Operations | arithmetic, `mean`/`std`, `relu`, `@` matrix multiply |
| 3 - Reshaping | `reshape`/`view` (and when `view` fails), `transpose`/`permute`, `flatten`, `squeeze` |
| 4 - Indexing + Broadcasting | slicing, boolean masks, the bias-add broadcast pattern |
| 5 - Autograd | `requires_grad`, `.backward()`, `.grad`, `torch.no_grad()`, hand-rolled gradient descent |

### Production hygiene (carry these forward)

- At inference, wrap the forward pass in `with torch.no_grad():` (or the newer, faster `torch.inference_mode()`) so you do not build a graph you will throw away.
- Use `.detach()` or `.item()` when you log a loss, so you do not accidentally keep the whole graph alive.
- Keep your model and your data on the SAME device, or you get a device-mismatch error.

### Self-check

1. Why must the argument to `.backward()` be a scalar, and how do you make it one?
2. Why does `.view()` fail right after `.transpose()`, and what are two fixes?
3. What happens if you forget to zero gradients inside a training loop?

### Next: B5 - Word2Vec and Sentence Embeddings

You can now build and differentiate a tensor expression. Next we turn real sentences into the 384-dimensional float tensors (from `all-MiniLM-L6-v2`) that this exact machinery will train on. The gradient descent you just ran by hand is what `optimizer.step()` will automate when we build the network in B6 and the word2vec MLP in B7.